# Dixon & Robinson (1998) モデル

Dixon & Robinson (1998) のモデルVIを実装して、StatsBombのデータに当てはめる。退場者の効果は入れず、得点だけを使う。

モデルVIに入っているもの
- チームごとの攻撃力 α と守備力 β
- ホームアドバンテージ γ_h
- 前半・後半の終わりの補正 ρ1, ρ2
- スコアによる得点強度の変化 λ_xy, μ_xy
- 時間とともに得点強度が増える効果 ξ1, ξ2

出力は各パラメータの推定値、対数尤度、AIC、BIC。

参考：Dixon, M. J. and Robinson, M. E. (1998). A birth process model for association football matches. *The Statistician*, 47(3), 523–538.

## データ

`*_goals_and_red_cards.csv` を使う。主な列は次のとおり。

| 列名 | 内容 |
|---|---|
| `match_id` | 試合ID |
| `home_team` / `away_team` | ホーム / アウェイのチーム |
| `home_away` | 得点したチーム（0=ホーム, 1=アウェイ） |
| `red_card` | 退場したチーム（今回は使わない） |
| `dr_time` | 試合時間を0〜1にした時刻 |

0-0の試合は得点の行がなく、`dr_time` が空の行が1行だけ入っている。0-0の試合も尤度に入れる必要があるので、「得点なしの試合」として扱う。

In [10]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize


## モデル

ホームの得点強度を $\lambda(t)$、アウェイの得点強度を $\mu(t)$ とする（$t$ は0〜1の試合時間、$i$ がホーム、$j$ がアウェイ）。

$$\lambda(t) = \rho(t)\big(\alpha_i\beta_j\gamma_h \cdot \lambda_{xy} + \xi_1 t\big)$$
$$\mu(t) = \rho(t)\big(\alpha_j\beta_i \cdot \mu_{xy} + \xi_2 t\big)$$

- $\lambda_{xy}, \mu_{xy}$ はスコアが $x$-$y$ のときの倍率（同点を1とする）
- $\rho(t)$ は前半の最後の1分が $\rho_1$、後半の最後の1分が $\rho_2$、それ以外は1

1試合の尤度は次のようになる（$J_l$ は $l$ 番目の得点がアウェイなら1、ホームなら0）。

$$L = \exp\Big(-\int_0^1 \lambda(t)\,dt\Big)\exp\Big(-\int_0^1 \mu(t)\,dt\Big)\prod_l \lambda(t_l)^{1-J_l}\,\mu(t_l)^{J_l}$$

攻撃力は平均が1になるようにする（$\frac{1}{n}\sum_i \alpha_i = 1$）。

### スコアの区分
| 区分 | スコア |
|---|---|
| level（基準） | 0-0, 1-1, 2-2, … |
| home1 | 1-0 |
| away1 | 0-1 |
| homeBig | 1-0以外でホームがリード |
| awayBig | 0-1以外でアウェイがリード |

In [11]:
# 前半・後半の最後の1分（ρ1, ρ2 を掛ける区間）
INJ1_START, INJ1_END = 44 / 90, 45 / 90
INJ2_START, INJ2_END = 89 / 90, 90 / 90


class _InvalidRate(Exception):
    """得点強度が0以下になったときに出すエラー"""
    pass


def _score_state(x, y):
    """スコア (x, y) をモデルVIの5区分（level, home1, away1, homeBig, awayBig）に分ける"""
    if x == y:
        return "level"
    if (x, y) == (1, 0):
        return "home1"
    if (x, y) == (0, 1):
        return "away1"
    if x > y:
        return "homeBig"
    return "awayBig"


## データの前処理

csvを、試合ごとの得点の並びに変える。退場の行は使わないので飛ばす。

In [12]:
def prepare_matches(df):
    """csvを試合ごとの得点の並びに変える。
    戻り値: [{"home", "away", "events": [(時刻, "H_GOAL" か "A_GOAL"), ...]}, ...]
    """
    matches = {}
    d = df.sort_values(["match_id", "dr_time"])
    for row in d.itertuples(index=False):
        mid = row.match_id
        if mid not in matches:
            matches[mid] = {"home": row.home_team, "away": row.away_team, "events": []}
        if pd.isna(row.dr_time):
            # 0-0の試合の行（得点なし）
            continue
        if pd.notna(row.red_card):
            # 退場の行は使わない
            continue
        t = float(row.dr_time)
        side = int(row.home_away)  # 得点したチーム（0=home, 1=away）
        kind = "H_GOAL" if side == 0 else "A_GOAL"
        matches[mid]["events"].append((t, kind))
    return list(matches.values())


def build_index(matches):
    teams = sorted(set([m["home"] for m in matches] + [m["away"] for m in matches]))
    idx = {t: i for i, t in enumerate(teams)}
    return teams, idx


## 尤度関数

パラメータをまとめたベクトル `theta` から各パラメータを取り出して、対数尤度を計算する。
正の値でないといけないパラメータ（α, β, γ_h, ρ, λ_xy, μ_xy）は `exp()` をかけて正にする。ξ はそのまま使う。
得点強度が0以下になるパラメータには、大きなペナルティを付ける。

In [13]:
PARAM_NAMES_GLOBAL = [
    "gamma_h", "rho1", "rho2",
    "lambda_home1", "lambda_away1", "lambda_homeBig", "lambda_awayBig",
    "mu_home1", "mu_away1", "mu_homeBig", "mu_awayBig",
    "xi1", "xi2",
]
N_GLOBAL = len(PARAM_NAMES_GLOBAL)


def unpack_params(theta, n_teams):
    a = theta[0:n_teams]
    b = theta[n_teams:2 * n_teams]
    g = theta[2 * n_teams:]

    alpha = np.exp(a)
    alpha = alpha / alpha.mean()      # 攻撃力の平均を1にする
    beta = np.exp(b)

    gamma_h = np.exp(g[0])
    rho1 = np.exp(g[1])
    rho2 = np.exp(g[2])
    lam = {"home1": np.exp(g[3]), "away1": np.exp(g[4]),
           "homeBig": np.exp(g[5]), "awayBig": np.exp(g[6]), "level": 1.0}
    mu = {"home1": np.exp(g[7]), "away1": np.exp(g[8]),
          "homeBig": np.exp(g[9]), "awayBig": np.exp(g[10]), "level": 1.0}
    xi1, xi2 = g[11], g[12]
    return alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2


In [14]:
def _check_pos(base, xi, t):
    if base + xi * t <= 0:
        raise _InvalidRate()


def _integrate_rate(t1, t2, base, xi, rho1, rho2):
    """区間 [t1, t2] で rho(t)*(base + xi*t) を積分する（rho が変わる時刻で区間を分ける）"""
    bpoints = sorted({t1, t2} | {p for p in (INJ1_START, INJ1_END, INJ2_START, INJ2_END) if t1 < p < t2})
    for p in bpoints:
        _check_pos(base, xi, p)
    total = 0.0
    for a, c in zip(bpoints[:-1], bpoints[1:]):
        mid = 0.5 * (a + c)
        if INJ1_START < mid <= INJ1_END:
            r = rho1
        elif INJ2_START < mid <= INJ2_END:
            r = rho2
        else:
            r = 1.0
        total += r * (base * (c - a) + xi * (c ** 2 - a ** 2) / 2.0)
    return total


def _rate_at(t, base, xi, rho1, rho2):
    """時刻 t の強度 rho(t)*(base + xi*t)"""
    val = base + xi * t
    if val <= 0:
        raise _InvalidRate()
    if INJ1_START < t <= INJ1_END:
        r = rho1
    elif INJ2_START < t <= INJ2_END:
        r = rho2
    else:
        r = 1.0
    return r * val


def _match_loglik(match, idx, alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2):
    hi, aj = idx[match["home"]], idx[match["away"]]
    lam_k = alpha[hi] * beta[aj] * gamma_h   # ホームの基礎の強度（同点のとき）
    mu_k = alpha[aj] * beta[hi]              # アウェイの基礎の強度（同点のとき）

    x = y = 0     # 今のスコア
    t_prev = 0.0
    ll = 0.0

    for t_ev, kind in sorted(match["events"], key=lambda e: e[0]):
        s_state = _score_state(x, y)

        base_h = lam_k * lam[s_state]
        base_a = mu_k * mu[s_state]

        # 前の得点から今の得点までの区間の積分を引く
        ll -= _integrate_rate(t_prev, t_ev, base_h, xi1, rho1, rho2)
        ll -= _integrate_rate(t_prev, t_ev, base_a, xi2, rho1, rho2)

        if kind == "H_GOAL":
            ll += np.log(_rate_at(t_ev, base_h, xi1, rho1, rho2))
            x += 1
        elif kind == "A_GOAL":
            ll += np.log(_rate_at(t_ev, base_a, xi2, rho1, rho2))
            y += 1
        t_prev = t_ev

    # 最後の得点から試合終了(t=1)までの区間
    s_state = _score_state(x, y)
    base_h = lam_k * lam[s_state]
    base_a = mu_k * mu[s_state]
    ll -= _integrate_rate(t_prev, 1.0, base_h, xi1, rho1, rho2)
    ll -= _integrate_rate(t_prev, 1.0, base_a, xi2, rho1, rho2)
    return ll


def negative_log_likelihood(theta, matches, idx):
    n_teams = len(idx)
    alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2 = unpack_params(theta, n_teams)
    total = 0.0
    for m in matches:
        try:
            total += _match_loglik(m, idx, alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2)
        except _InvalidRate:
            total += -1e9   # 強度が負になるときは大きなペナルティ
    if not np.isfinite(total):
        return 1e12
    return -total


def _count_invalid_matches(theta, matches, idx):
    """推定後のパラメータで、強度が0以下になる試合の数を数える"""
    n_teams = len(idx)
    alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2 = unpack_params(theta, n_teams)
    n_invalid = 0
    for m in matches:
        try:
            _match_loglik(m, idx, alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2)
        except _InvalidRate:
            n_invalid += 1
    return n_invalid


## 初期値

チームごとの平均得点・平均失点から、攻撃力と守備力の初期値を作る。

In [15]:
def initial_theta(matches, idx):
    n_teams = len(idx)
    goals_for = np.zeros(n_teams)
    goals_against = np.zeros(n_teams)
    games = np.zeros(n_teams)
    total_goals = 0
    total_games = 0
    home_goals_total = 0
    away_goals_total = 0
    for m in matches:
        hi, aj = idx[m["home"]], idx[m["away"]]
        xg = sum(1 for t, k in m["events"] if k == "H_GOAL")
        yg = sum(1 for t, k in m["events"] if k == "A_GOAL")
        goals_for[hi] += xg
        goals_against[aj] += xg
        goals_for[aj] += yg
        goals_against[hi] += yg
        games[hi] += 1
        games[aj] += 1
        total_goals += xg + yg
        total_games += 1
        home_goals_total += xg
        away_goals_total += yg

    avg = total_goals / max(2 * total_games, 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        att0 = np.where(games > 0, (goals_for / np.maximum(games, 1)) / avg, 1.0)
        def0 = np.where(games > 0, (goals_against / np.maximum(games, 1)) / avg, 1.0)
    att0 = np.clip(att0, 0.3, 3.0)
    def0 = np.clip(def0, 0.3, 3.0)

    a0 = np.log(att0)
    b0 = np.log(def0)

    gamma0 = home_goals_total / max(away_goals_total, 1)
    g0 = np.zeros(N_GLOBAL)
    g0[0] = np.log(max(gamma0, 0.1))   # gamma_h 以外は 0（倍率1）から始める
    return np.concatenate([a0, b0, g0])


## 推定

パラメータが多い（20チームなら53個）ので、2段階で最適化する。

1. L-BFGS-B でおおまかに合わせる
2. その結果から Powell 法で仕上げる

In [19]:
def fit_dixon_robinson(df, maxiter=1000, disp=False):
    """モデルVIを推定する。

    戻り値:
        summary       : 対数尤度・AIC・BICなど
        team_params   : チームごとの攻撃力・守備力
        global_params : チーム以外のパラメータ
        res           : 最適化の結果
    """
    matches = prepare_matches(df)
    teams, idx = build_index(matches)
    n_teams = len(teams)

    theta0 = initial_theta(matches, idx)

    # パラメータが動ける範囲（xi 以外は log の値）
    bounds = [(-3, 3)] * n_teams + [(-3, 3)] * n_teams
    bounds += [(-3, 3)]            # gamma_h
    bounds += [(-3, 3), (-3, 3)]   # rho1, rho2
    bounds += [(-3, 3)] * 4        # lambda_home1, lambda_away1, lambda_homeBig, lambda_awayBig
    bounds += [(-3, 3)] * 4        # mu_home1, mu_away1, mu_homeBig, mu_awayBig
    bounds += [(-5, 5), (-5, 5)]   # xi1, xi2（log ではない）

    res1 = minimize(
        negative_log_likelihood, theta0, args=(matches, idx),
        method="L-BFGS-B", bounds=bounds,
        options={"maxiter": maxiter, "maxfun": maxiter * 50},
    )
    res = minimize(
        negative_log_likelihood, res1.x, args=(matches, idx),
        method="Powell", bounds=bounds,
        options={"maxiter": maxiter * 20, "maxfev": maxiter * 200, "xtol": 1e-10, "ftol": 1e-12},
    )
    if disp:
        print(f"[stage1: L-BFGS-B] loglik={-res1.fun:.4f} success={res1.success}")
        print(f"[stage2: Powell]   loglik={-res.fun:.4f} success={res.success}")

    theta = res.x
    alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2 = unpack_params(theta, n_teams)

    n_params = len(theta)
    loglik = -res.fun
    aic = 2 * n_params - 2 * loglik
    # BIC の n は試合数にする
    bic = n_params * np.log(len(matches)) - 2 * loglik

    team_params = pd.DataFrame({"team": teams, "alpha_attack": alpha, "beta_defence": beta})
    global_params = pd.DataFrame({
        "parameter": [
            "gamma_h", "rho1", "rho2",
            "lambda_home1", "lambda_away1", "lambda_homeBig", "lambda_awayBig",
            "mu_home1", "mu_away1", "mu_homeBig", "mu_awayBig",
            "xi1", "xi2",
        ],
        "estimate": [
            gamma_h, rho1, rho2,
            lam["home1"], lam["away1"], lam["homeBig"], lam["awayBig"],
            mu["home1"], mu["away1"], mu["homeBig"], mu["awayBig"],
            xi1, xi2,
        ],
    })

    n_invalid = _count_invalid_matches(theta, matches, idx)
    message = str(res.message)
    if n_invalid > 0:
        message = (
        f"[WARNING] Even after convergence, {n_invalid} match(es) still have "
        f"non-positive scoring intensity, so a -1e9 penalty has leaked into "
        f"the log-likelihood. The log-likelihood/AIC/BIC values are not "
        f"reliable. " + message
    )

    summary = {
        "n_matches": len(matches), "n_teams": n_teams, "n_params": n_params,
        "log_likelihood": loglik, "AIC": aic, "BIC": bic,
        "converged": bool(res.success), "message": message,
    }

    print("==== モデル適合結果 ====")
    print(f"試合数: {summary['n_matches']}, チーム数: {summary['n_teams']}, パラメータ数: {summary['n_params']}")
    print(f"対数尤度: {summary['log_likelihood']:.3f}")
    print(f"AIC: {summary['AIC']:.3f}")
    print(f"BIC: {summary['BIC']:.3f}")
    print(f"収束: {summary['converged']} ({summary['message']})")
    print()
    print("---- チーム別パラメータ ----")
    print(team_params.to_string(index=False))
    print()
    print("---- 共通パラメータ ----")
    print(global_params.to_string(index=False))

    return summary, team_params, global_params, res


## 1ファイルで試す

Premier League 2015/16 で推定してみる。

In [20]:
df = pd.read_csv("../statsbomb_data/Premier_League/PL2015-2016_goals_and_red_cards.csv")
summary, team_params, global_params, res = fit_dixon_robinson(df, disp=True)


[stage1: L-BFGS-B] loglik=-555.5935 success=True
[stage2: Powell]   loglik=-555.5935 success=True
==== モデル適合結果 ====
試合数: 380, チーム数: 20, パラメータ数: 53
対数尤度: -555.594
AIC: 1217.187
BIC: 1426.016
収束: True (Optimization terminated successfully.)

---- チーム別パラメータ ----
                team  alpha_attack  beta_defence
     AFC Bournemouth      0.853595      1.108864
             Arsenal      1.340754      0.514741
         Aston Villa      0.347676      1.239235
             Chelsea      1.218281      0.774309
      Crystal Palace      0.670022      0.762341
             Everton      1.231369      0.857065
      Leicester City      1.398502      0.450133
           Liverpool      1.299154      0.726263
     Manchester City      1.516648      0.652527
   Manchester United      0.936050      0.482139
    Newcastle United      0.792760      1.050016
        Norwich City      0.711203      1.058151
         Southampton      1.170900      0.557423
          Stoke City      0.781877      0.817060
     

## 全リーグ・全シーズンで推定する

`statsbomb_data` の全ファイルで推定し、AIC・BICの一覧とパラメータをcsvに保存する。

In [21]:
import glob
import os

DATA_DIR = "../statsbomb_data"
OUT_DIR = "./dixon_robinson_pure_results"
os.makedirs(OUT_DIR, exist_ok=True)

csv_paths = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_goals_and_red_cards.csv"), recursive=True))
print(f"{len(csv_paths)} 件のcsvが見つかりました")

results_all = []
for path in csv_paths:
    print("=" * 60)
    print(path)
    df_i = pd.read_csv(path)
    league = df_i["competition_name"].iloc[0] if "competition_name" in df_i.columns and len(df_i) else os.path.basename(path)
    season = df_i["season_name"].iloc[0] if "season_name" in df_i.columns and len(df_i) else ""
    tag = f"{league}_{season}".replace("/", "-").replace(" ", "_")
    try:
        summary_i, team_params_i, global_params_i, _ = fit_dixon_robinson(df_i, disp=True)
        summary_i["league"] = league
        summary_i["season"] = season
        summary_i["file"] = os.path.basename(path)
        results_all.append(summary_i)
        team_params_i.to_csv(os.path.join(OUT_DIR, f"team_params_{tag}.csv"), index=False)
        global_params_i.to_csv(os.path.join(OUT_DIR, f"global_params_{tag}.csv"), index=False)
    except Exception as e:
        print("失敗:", e)

summary_all_df = pd.DataFrame(results_all)
summary_all_df.to_csv(os.path.join(OUT_DIR, "summary_all.csv"), index=False)
summary_all_df


11 件のcsvが見つかりました
../statsbomb_data/FA_Women's_Super_League/WSL2018-2019_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-99.5337 success=True
[stage2: Powell]   loglik=-99.5337 success=True
==== モデル適合結果 ====
試合数: 107, チーム数: 11, パラメータ数: 35
対数尤度: -99.534
AIC: 269.067
BIC: 362.616
収束: True (Optimization terminated successfully.)

---- チーム別パラメータ ----
                      team  alpha_attack  beta_defence
               Arsenal WFC      2.282710      0.686594
       Birmingham City WFC      0.954218      0.830367
Brighton & Hove Albion WFC      0.558189      1.732132
          Bristol City WFC      0.572884      1.622833
               Chelsea FCW      1.321441      0.633739
               Everton LFC      0.524138      1.591930
             Liverpool WFC      0.709887      1.731127
       Manchester City WFC      1.726996      0.856166
               Reading WFC      1.161912      1.504190
       West Ham United LFC      0.802755      1.646773
           Yeovil Town LFC      0.384871    

,n_matches,n_teams,n_params,log_likelihood,AIC,BIC,converged,message,league,season,file
0,107,11,35,-9.953372e+01,2.690674e+02,3.626164e+02,True,Optimization terminated successfully.,FA Women's Super League,2018/2019,WSL2018-2019_goals_and_red_cards.csv
1,87,12,37,-6.041685e+01,1.948337e+02,2.860723e+02,True,Optimization terminated successfully.,FA Women's Super League,2019/2020,WSL2019-2020_goals_and_red_cards.csv
2,131,12,37,-9.335717e+01,2.607143e+02,3.670966e+02,True,Optimization terminated successfully.,FA Women's Super League,2020/2021,WSL2020-2021_goals_and_red_cards.csv
3,132,12,37,-7.978453e+01,2.335691e+02,3.402327e+02,True,Optimization terminated successfully.,FA Women's Super League,2023/2024,WSL2023-2024_goals_and_red_cards.csv
4,132,12,37,-1.000000e+09,2.000000e+09,2.000000e+09,True,"[WARNING] Even after convergence, 1 match(es) ...",Frauen Bundesliga,2023/2024,FB2023-2024_goals_and_red_cards.csv
5,115,11,35,-1.218186e+02,3.136372e+02,4.097098e+02,True,Optimization terminated successfully.,Indian Super league,2021/2022,ISL2021-2022_goals_and_red_cards.csv
6,380,20,53,-5.222177e+02,1.150435e+03,1.359264e+03,True,Optimization terminated successfully.,La Liga,2015/2016,LL2015-2016_goals_and_red_cards.csv
7,240,16,45,-1.622868e+02,4.145735e+02,5.712023e+02,True,Optimization terminated successfully.,Liga F,2023/2024,LF2023-2024_goals_and_red_cards.csv
8,377,20,53,-5.810269e+02,1.268054e+03,1.476463e+03,True,Optimization terminated successfully.,Ligue 1,2015/2016,L12015-2016_goals_and_red_cards.csv
9,380,20,53,-5.555935e+02,1.217187e+03,1.426016e+03,True,Optimization terminated successfully.,Premier League,2015/2016,PL2015-2016_goals_and_red_cards.csv
